[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmcguir2/astromol/blob/refactor/docs/notebooks/04_tables_and_slides.ipynb)

# Generate tables and slides

This notebook writes LaTeX table fragments and PowerPoint slides from a census view.

Set `VIEW_CHOICE = "current"` below when you want the latest live database outputs.

In [ ]:
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/bmcguir2/astromol.git@refactor",
    ])

In [ ]:
from pathlib import Path

from astromol.census import CensusView
from astromol.database import Database
from astromol.latex import (
    write_exgal_table,
    write_ism_tables,
    write_ppd_table,
    write_scalar_fragments,
)
from astromol.slides import (
    build_molecule_slide_layout,
    write_molecule_slide,
    write_molecule_slide_report,
    write_ppd_detection_slide,
)

# Use "current" for the latest live database, "2026" for the developing
# 2026 census view, or "2021" for historical reproduction.
VIEW_CHOICE = "current"

db = Database()
if VIEW_CHOICE == "current":
    view = CensusView.current(db)
    view_label = "current"
else:
    view = CensusView.for_census(db, VIEW_CHOICE)
    view_label = VIEW_CHOICE

out = Path("astromol_table_slide_outputs")
table_dir = out / "tables"
slide_dir = out / "slides"
table_dir.mkdir(parents=True, exist_ok=True)
slide_dir.mkdir(parents=True, exist_ok=True)

print(f"Using {view_label!r} view")

## Write LaTeX fragments

Writer functions return a dictionary of `{filename: content}` and also write the files to disk.

In [ ]:
write_scalar_fragments(view, table_dir)
write_ism_tables(view, table_dir, layout="balanced")
write_exgal_table(view, table_dir)
write_ppd_table(view, table_dir)

for path in sorted(table_dir.glob("*.tex")):
    print(path)

## Build a slide layout report

The report is useful before writing or manually inspecting a PowerPoint file.

In [ ]:
layout = build_molecule_slide_layout(view, profile="balanced")
report_path = write_molecule_slide_report(
    layout,
    slide_dir / f"astro_molecules_{view_label}_layout_report.md",
)

print(report_path.read_text())

## Write PowerPoint slides

The ISM/CSM slide uses the balanced profile for current-census production. The PPD slide includes isotopologues by default.

In [ ]:
ism_slide = write_molecule_slide(
    view,
    slide_dir / f"astro_molecules_{view_label}.pptx",
    profile="balanced",
)

ppd_slide = write_ppd_detection_slide(
    view,
    slide_dir / f"ppd_molecules_{view_label}.pptx",
)

print(ism_slide)
print(ppd_slide)

## Download files from Colab

If you are running in Colab, the following cell downloads the generated PowerPoint files to your machine.

In [ ]:
if IN_COLAB:
    from google.colab import files  # type: ignore

    files.download(str(ism_slide))
    files.download(str(ppd_slide))
else:
    print("Not running in Colab; files are available under", slide_dir)